# Compile quantum circuits into pulses
In this first exercise, you will get an idea of unitary gate implementation through direct visualization of pulses.

In this notebook, you will ...
* Prepare a GHZ state and compile the corresponding quantum circuit into a *playlist* of microwave pulses.

By the end of this notebook, you will understand the correspondence between unitary gates and their physical implementation for superconducting qubits.

In order to get started, make sure you have the appropriate packages installed:

In [1]:
!pip install pip==23.0

In [2]:
# %%capture
!pip install iqm-client==32.1.1
!pip install iqm-station-control-client==11.3.1
!pip install iqm-pulla==11.16.2
!pip install iqm-pulse==12.6.1

In [3]:
!pip install qrisp[iqm]

zsh:1: no matches found: qrisp[iqm]


## 1. Preparation

### 1.1 Connecting to the QPU station control
In the following, we will use the PulLa (Pulse Level Access) package. You can find the documentation [here](https://docs.meetiqm.com/iqm-pulla/).

As a first step, we need to create a **PulLa object**. In general, this is a **compiler**, in a particular state, linked to a particular quantum computer. It contains calibration data and the set of available operations.

Make sure you have the correct url and token below.

In [4]:
from iqm.pulla.pulla import Pulla

api_token = "kZRhVQdEjZNigffaT99achQVrvPc0+mbT3XnI8ghm5EBmt6h2wZyM5KFVSOHGqk0"
p = Pulla("https://cocos.resonance.meetiqm.com/garnet", get_token_callback=lambda: api_token)

compiler = p.get_standard_compiler()

/Users/haslab-test/Desktop/iqm/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. GHZ state

You have learnt in the past labs and exercises what a GHZ state is and how to prepare one on a quantum computer. Here, we are not interested in its entanglement properties, but we will use it as a starting point to look into physical pulses. Let's construct a GHZ for 3 qubits:

In [5]:
from qrisp import QuantumCircuit

qc_0 = QuantumCircuit(3)
qc_0.h(0)
qc_0.cx(0, 1)
qc_0.cx(0, 2)
qc_0.measure([0,1,2])

print(qc_0)

       ┌───┐             ┌─┐   
qb_58: ┤ H ├──■────■─────┤M├───
       └───┘┌─┴─┐  │  ┌─┐└╥┘   
qb_59: ─────┤ X ├──┼──┤M├─╫────
            └───┘┌─┴─┐└╥┘ ║ ┌─┐
qb_60: ──────────┤ X ├─╫──╫─┤M├
                 └───┘ ║  ║ └╥┘
 cb_0: ════════════════╬══╩══╬═
                       ║     ║ 
 cb_1: ════════════════╩═════╬═
                             ║ 
 cb_2: ══════════════════════╩═
                               


Before compiling the circuit, we need to transpile it to take into account the native gates and connectivity of IQM quantum computers:

In [6]:
from iqm.qiskit_iqm import IQMProvider

provider = IQMProvider('https://cocos.resonance.meetiqm.com/garnet', token=api_token)
backend = provider.get_backend()

transpiled_qc = qc_0.transpile(backend = backend)
print(transpiled_qc)

                                                                     
 ancilla.0: ─────────────────────────────────────────────────────────
                                                                     
 ancilla.1: ─────────────────────────────────────────────────────────
            ┌─────────────┐   ┌─────────────┐                  ┌─┐   
     qb_59: ┤ R(π/2,3π/2) ├─■─┤ R(π/2,5π/2) ├──────────────────┤M├───
            └─────────────┘ │ └─────────────┘                  └╥┘   
 ancilla.2: ────────────────┼───────────────────────────────────╫────
                            │                                   ║    
 ancilla.3: ────────────────┼───────────────────────────────────╫────
                            │                                   ║    
 ancilla.4: ────────────────┼───────────────────────────────────╫────
                            │                                   ║    
 ancilla.5: ────────────────┼───────────────────────────────────╫────
            ┌───────

To use the PulLA visualization tool, we need to convert the qrisp circuit into a qiskit one. We use the converter as below:

In [7]:
qiskit_circuit = transpiled_qc.to_qiskit()

## 3. Compilation

We can now extract the list of *instructions* from the circuit defined above and compile it to obtain a *playlist* of pulses!

In [8]:
from iqm.qiskit_iqm.qiskit_to_iqm import serialize_instructions
from iqm.iqm_client import Circuit

iqm_instructions = serialize_instructions(qiskit_circuit, {i : "QB" + str(i+1) for i in range(qiskit_circuit.num_qubits)})
iqm_circuit = Circuit(name = "my_circuit", instructions = iqm_instructions)

playlist, context = compiler.compile([iqm_circuit])

Finally, we are ready to visualize the pulses that prepare a GHZ state of 3 qubits. Let's explore the details:

In [9]:
from iqm.pulse.playlist.visualisation.base import inspect_playlist
from IPython.core.display import HTML

HTML(inspect_playlist(playlist))

/Users/haslab-test/Desktop/iqm/myenv/lib/python3.11/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
